# Phase 3.2 Notebook: Entity Deduplication
Reads disambiguation output and writes only to data/31_entity_deduplication.
This phase is manual-only for merge decisions.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

In [ ]:
from process.entity_disambiguation.config import INPUT_PHASE_DIR as P3A_DIR, FILE_SELECTIONS as P3A_SELECTIONS
from process.entity_disambiguation.ingestion import ingest_selected_candidates
from process.entity_deduplication.config import (
    PHASE_DIR, FILE_ENTITY_SCORES, FILE_MERGE_RECOMMENDATIONS, FILE_MERGE_SELECTIONS, MERGE_SELECTION_COLUMNS
)
from process.entity_deduplication.similarity import score_entity_pairs
from process.entity_deduplication.recommendation import build_merge_recommendations
from process.entity_deduplication.selection import build_review_sheet, require_manual_gate
from process.entity_deduplication.merging import apply_merge_decisions

# Ensure P3a manual decisions are available and accepted entries are loaded.
accepted = ingest_selected_candidates()
candidates = pd.read_csv(ROOT / P3A_DIR.parent / '20_canidate_generation' / 'candidates.csv')

entities = accepted.merge(candidates[['candidate_id', 'candidate_label']], on='candidate_id', how='left')
entities = entities[['candidate_id', 'candidate_label']].dropna().drop_duplicates().rename(
    columns={'candidate_id': 'entity_id', 'candidate_label': 'label'}
)

scores = score_entity_pairs(entities, min_score=0.9)
recs = build_merge_recommendations(scores)
review = build_review_sheet(scores, recs)

(ROOT / PHASE_DIR).mkdir(parents=True, exist_ok=True)
scores.to_csv(ROOT / PHASE_DIR / FILE_ENTITY_SCORES, index=False)
recs.to_csv(ROOT / PHASE_DIR / FILE_MERGE_RECOMMENDATIONS, index=False)
review.to_csv(ROOT / PHASE_DIR / 'review_sheet.csv', index=False)
entities.to_csv(ROOT / PHASE_DIR / 'entities_input.csv', index=False)

print(f'entities: {len(entities)}')
print(f'scores: {len(scores)}')
print(f'recommendations: {len(recs)}')

In [ ]:
merge_path = ROOT / PHASE_DIR / FILE_MERGE_SELECTIONS
if not merge_path.exists():
    template = recs[['entity_id_left', 'entity_id_right']].copy()
    template['decision'] = ''
    template['merged_entity_id'] = ''
    template['reason'] = ''
    template['reviewer'] = ''
    template['reviewed_at'] = ''
    template = template[MERGE_SELECTION_COLUMNS]
    template.to_csv(merge_path, index=False)
    print(f'Created manual template: {merge_path}')
else:
    print(f'Manual decisions file exists: {merge_path}')

In [ ]:
try:
    _ = require_manual_gate(ROOT / PHASE_DIR)
    canonical = apply_merge_decisions(entities)
    canonical.to_csv(ROOT / PHASE_DIR / 'entities.csv', index=False)
    print(f'Manual gate passed. Canonical entities saved: {len(canonical)}')
except Exception as exc:
    print('Manual gate not passed yet. Fill selections.csv and rerun this cell.')
    print(str(exc))